제출 방법: 본 코랩 노트를 파일 > 드라이브에 사본 저장 후 빈칸 ____ 부분을 채워 실행한 뒤 제출하세요.

과제 목표:

- U-Net의 핵심인 Skip Connection(Feature Map 결합) 동작 원리를 이해합니다.

- ViT의 핵심인 Patch Embedding과 [CLS] Token & Position Embedding 결합 방식을 이해합니다.

---
## Task 1. U-Net의 핵심: Skip Connection 구현하기
U-Net은 Contracting Path(인코더)에서 얻은 지역적 특징(Feature Map)을 Expanding Path(디코더)로 전달하여 연결하는 Skip Connection을 사용합니다.

빈칸 요소:

Up-sampling 레이어 (nn.ConvTranspose2d)

Feature Map 결합 함수 및 대상 (torch.cat, x_encoder)

In [5]:
import torch
import torch.nn as nn

In [6]:
class UNetSkipConnectionBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # [빈칸 1] 업샘플링을 위한 역합성곱(Transposed Convolution) 레이어 선언
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)

        self.conv = nn.Sequential(
            nn.Conv2d(out_channels * 2, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x_decoder, x_encoder):
        x_up = self.up(x_decoder)

        # [빈칸 2] Skip Connection: PyTorch의 결합 함수와 건너온 인코더 특성(x_encoder) 연결
        x_cat = torch.cat([x_up, x_encoder], dim=1)

        output = self.conv(x_cat)
        return output

# --- 검증 코드 ---
# 배치 크기 2, 채널 128, 크기 16x16
x_dec = torch.randn(2, 128, 16, 16)
# Skip connection으로 연결될 인코더 특성 (채널 64, 크기 32x32)
x_enc = torch.randn(2, 64, 32, 32)

unet_block = UNetSkipConnectionBlock(in_channels=128, out_channels=64)
out = unet_block(x_dec, x_enc)

print("U-Net 출력 Shape:", out.shape)
assert out.shape == torch.Size([2, 64, 32, 32]), "Task 1 정답이 아닙니다. 차원을 다시 확인하세요!"
print("Task 1 성공!\n")

U-Net 출력 Shape: torch.Size([2, 64, 32, 32])
Task 1 성공!



## Task 2. ViT의 핵심: Patch Embedding & Class Token 구현하기

ViT는 이미지를 $16 \times 16$ 패치로 나누어 1D 벡터로 바꾼 후, 맨 앞에 학습 가능한 [CLS] 토큰을 붙이고 Position Embedding을 더해줍니다.

빈칸 요소:

차원 변형 메소드 (flatten, transpose)

학습 가능 파라미터 지정 (nn.Parameter)

CLS 토큰 결합 대상 (cls_tokens)

In [7]:
class ViTPatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.projection = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

        # [빈칸 3] 역전파 시 학습이 가능하도록 텐서를 모델 파라미터로 등록하는 PyTorch 클래스
        # Hint: nn.Parameter(...)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, 1 + self.num_patches, embed_dim))

    def forward(self, x):
        B = x.shape[0]

        x = self.projection(x) # [B, embed_dim, 14, 14]

        # [빈칸 4] 2D 공간 차원을 1D로 평평하게 펴고(flatten), 차원 축을 교환(transpose)합니다.
        x = x.flatten(2).transpose(1, 2)

        cls_tokens = self.cls_token.expand(B, -1, -1)

        # [빈칸 5] CLS 토큰과 패치 임베딩 x를 시퀀스 차원(dim=1) 기준으로 연결
        x = torch.cat([cls_tokens, x], dim=1)

        # [빈칸 6] Position Embedding 가산
        x = x + self.pos_embedding

        return x

# --- 검증 코드 ---
# 배치 크기 2, 3채널(RGB), 224x224 이미지
x_img = torch.randn(2, 3, 224, 224)

vit_embed = ViTPatchEmbedding(img_size=224, patch_size=16, embed_dim=768)
vit_out = vit_embed(x_img)

print("ViT Embedding 출력 Shape:", vit_out.shape)
assert vit_out.shape == torch.Size([2, 197, 768]), "Task 2 정답이 아닙니다. 빈칸을 다시 확인하세요!"
print("Task 2 성공!")

ViT Embedding 출력 Shape: torch.Size([2, 197, 768])
Task 2 성공!


## 📌 과제 개념 정리

### Task 1. U-Net — Skip Connection

U-Net은 이미지를 압축(인코더) → 복원(디코더)하는 구조이다.
압축 과정에서 디테일 정보가 손실되므로, 인코더 단계의 feature map을 디코더로 직접 전달해 결합하는 것이 **Skip Connection**이다.

- `nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)`
  → 디코더의 작은 feature map을 **2배로 업샘플링**(확대)하는 레이어. 일반 Conv2d와 반대로 크기를 키운다.

- `torch.cat([x_up, x_encoder], dim=1)`
  → 업샘플링한 디코더 feature map과, 이전에 저장해둔 인코더 feature map을 **채널(dim=1) 방향으로 이어붙임**.
  → 64채널 + 64채널 = 128채널이 되고, 이후 `conv` 레이어가 다시 64채널로 정리한다.

**비유**: 흐릿하게 확대된 이미지(디코더) 위에, 원본의 선명한 스케치(인코더)를 겹쳐서 더 정확한 그림을 그리는 것.

---

### Task 2. ViT — Patch Embedding & CLS Token

ViT는 이미지를 16×16 패치로 잘라 "단어"처럼 취급하고, Transformer 구조로 처리한다.

- `nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)`
  → 이미지를 patch_size 단위로 잘라서 각 패치를 embed_dim 차원 벡터로 변환. (자르기 + 벡터화를 동시에 수행)

- `nn.Parameter(...)`
  → 텐서를 **학습 가능한 파라미터**로 등록. `cls_token`과 `pos_embedding`은 처음엔 랜덤값이지만 학습되면서 의미를 갖게 된다.
  - `cls_token`: 이미지 전체를 대표하는 "요약용" 토큰
  - `pos_embedding`: 패치들의 순서(위치) 정보를 부여하는 값

- `x.flatten(2).transpose(1, 2)`
  → Conv 출력 `[B, embed_dim, 14, 14]` (그리드 형태)을
  → `flatten(2)`로 `[B, embed_dim, 196]` (한 줄로 펴기)
  → `transpose(1, 2)`로 `[B, 196, embed_dim]` (Transformer가 원하는 "토큰 시퀀스" 형태)로 변환.

- `torch.cat([cls_tokens, x], dim=1)`
  → 패치 시퀀스 맨 앞에 cls_token을 붙여 `[B, 197, embed_dim]`으로 만듦 (196패치 + CLS 1개).

- `x + self.pos_embedding`
  → 각 토큰에 위치 정보를 **더해줌** (곱셈이 아님).

**비유**: 이미지를 퍼즐 조각(패치)으로 잘라 한 줄로 늘어놓고, 맨 앞에 "이 그림 전체 요약해줄게" 토큰을 붙인 다음, 각 조각에 "너는 몇 번째 자리야"라는 번호표(위치 정보)를 붙여주는 것.

---

### ✅ 검증 결과
- Task 1: `out.shape == [2, 64, 32, 32]` → 인코더와 동일한 크기로 복원됨
- Task 2: `vit_out.shape == [2, 197, 768]` → 배치 2, 토큰 197개(패치 196 + CLS 1), 768차원